### Muntatge de Google Drive
En aquesta cel·la es munta Google Drive per poder accedir als datasets, models i fitxers emmagatzemats al núvol des de l'entorn de Google Colab.


In [1]:
! pip install opencxr

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opencxr to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 47.9 MB/s eta 0:00:00
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=105612ab5ac74dcdfc58ac9538e810f2d7b3994826055cad87b6750242aa8752
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Definició de rutes del projecte
Es defineixen les rutes base del projecte, incloent els directoris d'imatges, metadades i models, per facilitar l'organització i reutilització del codi.


In [3]:
BASE_PATH = "/content/drive/MyDrive/ML/DATASETS/NODE21/proccessed_data"

PATH_IMAGES = f"{BASE_PATH}/images"
PATH_METADATA = f"{BASE_PATH}/metadata_augmented.csv"
PATH_METADATA = "/content/drive/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv"
#PATH_METADATA = "/content/drive/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata.csv"

print(PATH_IMAGES)
print(PATH_METADATA)

/content/drive/MyDrive/ML/DATASETS/NODE21/proccessed_data/images
/content/drive/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv


### Càrrega del fitxer de metadades
En aquesta cel·la es carrega el fitxer CSV amb la informació de les imatges (bounding boxes, etiquetes, rutes, etc.) en un DataFrame de pandas.


In [4]:
import pandas as pd
import os

df = pd.read_csv(PATH_METADATA)

df["file_path"] = df["img_name"].apply(
    lambda x: f"{PATH_IMAGES}/{x.replace('.mha', '.png')}"
)

df.head()
num_neg = (df["label"] == 0).sum()
num_pos = (df["label"] == 1).sum()
print("Negativos:", num_neg)
print("Positivos:", num_pos)

Negativos: 3748
Positivos: 4427


### Preparació del dataset NODE21 per a YOLO

Aquesta cel·la prepara el dataset NODE21 en format YOLO.  
Es defineixen les rutes, es carreguen les metadades, es divideixen les imatges en entrenament i validació evitant *data leakage*, i es converteixen les anotacions al format normalitzat de YOLO.  
Finalment, es copien les imatges i es generen els fitxers de labels corresponents.


In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# -----------------------------------
# RUTAS BASE
# -----------------------------------
BASE_DATA   = "/content/drive/MyDrive/ML/DATASETS/NODE21"
BASE_PATH   = f"{BASE_DATA}/proccessed_data"

IMG_SRC    = f"{BASE_PATH}/images"
CSV_PATH   = f"{BASE_PATH}/metadata_augmented_def2.csv"

YOLO_BASE  = f"{BASE_DATA}/NODE21_YOLO"

IMG_TRAIN = f"{YOLO_BASE}/images/train"
IMG_VAL   = f"{YOLO_BASE}/images/val"
LBL_TRAIN = f"{YOLO_BASE}/labels/train"
LBL_VAL   = f"{YOLO_BASE}/labels/val"

for p in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    os.makedirs(p, exist_ok=True)

# -----------------------------------
# CARGAR METADATA
# -----------------------------------
df = pd.read_csv(CSV_PATH)

df["file_path"] = df["img_name"].apply(
    lambda x: f"{IMG_SRC}/{x.replace('.mha', '.png')}"
)

df = df[df["file_path"].apply(os.path.exists)]

print(f"Imágenes totales: {df['file_path'].nunique()}")

# -----------------------------------
# SPLIT POR IMAGEN (ANTI-LEAKAGE)
# -----------------------------------
images = df["file_path"].unique()

train_imgs, val_imgs = train_test_split(
    images,
    test_size=0.2,
    random_state=42
)

split_map = {p: "train" for p in train_imgs}
split_map.update({p: "val" for p in val_imgs})

# -----------------------------------
# CONVERSIÓN A YOLO
# -----------------------------------
IMG_SIZE = 1024

for img_path in tqdm(images, desc="Convirtiendo a YOLO"):
    split = split_map[img_path]
    img_name = os.path.basename(img_path)

    # Copiar imagen
    dst_img = IMG_TRAIN if split == "train" else IMG_VAL
    shutil.copy(img_path, f"{dst_img}/{img_name}")

    # Crear label YOLO (aunque esté vacío)
    label_name = img_name.replace(".png", ".txt")
    dst_lbl = LBL_TRAIN if split == "train" else LBL_VAL
    label_path = f"{dst_lbl}/{label_name}"

    rows = df[df["file_path"] == img_path]

    with open(label_path, "w") as f:
        for _, r in rows.iterrows():
            if r["label"] != 1:
                continue

            x, y, w, h = r["x"], r["y"], r["width"], r["height"]

            cx = (x + w / 2) / IMG_SIZE
            cy = (y + h / 2) / IMG_SIZE
            w  = w / IMG_SIZE
            h  = h / IMG_SIZE

            f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

print("Conversión YOLO completada")


### Creació del fitxer de configuració YOLO

Aquesta cel·la genera el fitxer `node21.yaml`, que defineix la ruta del dataset, els conjunts d’entrenament i validació, i la configuració de classes necessària per entrenar el model YOLO.


In [ ]:
yaml = f"""
path: {YOLO_BASE}
train: images/train
val: images/val

nc: 1
names: ['nodule']
"""

with open(f"{YOLO_BASE}/node21.yaml", "w") as f:
    f.write(yaml)

print("node21.yaml creado")


In [10]:
!ls /content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO/images/train | wc -l
!ls /content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO/labels/train | wc -l
!ls /content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO/images/val | wc -l
!ls /content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO/labels/val | wc -l



5720
5720
1430
1430


In [7]:
pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.1 MB/s eta 0:00:00


In [13]:
!cat /content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO/node21.yaml


path: /content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO
train: images/train
val: images/val

nc: 1
names: ['nodule']


In [14]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0)

(True, 'NVIDIA A100-SXM4-80GB')

In [5]:
!ls /content/drive/MyDrive/ML/DATASETS/NODE21

ls: cannot access '/content/drive/MyDrive/ML/DATASETS/NODE21': No such file or directory


### Entrenament del model YOLOv8

En aquesta cel·la es carrega un model YOLOv8 preentrenat i s’entrena amb el dataset NODE21.  
Es defineixen els paràmetres d’entrenament, l’optimitzador, la mida d’imatge, les estratègies d’augment de dades i els criteris d’aturada anticipada.


In [14]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO/node21.yaml",
    imgsz=1024,
    epochs=100,
    batch=4,
    device=0,
    workers=2,

    optimizer="AdamW",
    lr0=1e-4,
    lrf=1e-5,

    box=7.5,
    cls=0.3,
    dfl=1.5,

    mosaic=0.0,
    mixup=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    fliplr=0.5,
    flipud=0.0,
    scale=0.1,
    translate=0.05,

    patience=20,
    project="NODE21_YOLO",
    name="yolov8s_1024_rx",
    exist_ok=True
)


Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.3, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ML/DATASETS/NODE21/NODE21_YOLO/node21.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=1e-05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=yolov8s_1024_rx, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_m

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79c388e32e70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [15]:
!cp -r /content/NODE21_YOLO /content/drive/MyDrive/ML/DATASETS/NODE21/
